In [ ]:
import os
import sys
from pathlib import Path
import duckdb
import toml

from loguru import logger
from horse_logic.motherduck import DuckDBManager

In [ ]:
# Customise the logging outputs

def stdout_filter(record):
    return record["level"].name == "INFO" and "ECHO" in record["message"]

def customise_logger(logger_filename="DuckDBManager.log", logger_size=10, filter_func=stdout_filter, level="INFO"):
    try:
        logger.remove(0) # Remove default output to sys.stderr
        logger.add(sys.stdout, filter=filter_func, level=level)     # Add sys.stdout logging of select messages
        logger.add(logger_filename, rotation=f"{logger_size} MB", level=level)  # logs will be written to file.log
        logger.info(f"ECHO: Logging to {logger_filename}: size = {logger_size} MB")
        logger.info("ECHO: Limited logging to stdout using `stdout_filter`")
    except Exception as e:
        logger.error(f"Error with customising logger: {e}")

In [ ]:
customise_logger()

In [ ]:
logger.info(f"DuckDB version: {duckdb.__version__}")

In [ ]:
SECRETS_TOML = Path.cwd().parent / ".secrets.toml"

In [ ]:
def set_md_token(secrets_file=SECRETS_TOML):
    """
    Load secrets from a file and set the motherduck_token environment variable.
    """
    if "motherduck_token" not in os.environ:
        try:
            secrets = toml.load(secrets_file)
            os.environ["motherduck_token"] = secrets["MD_TOKEN"]
            logger.info(f"ECHO: Secrets loaded from {secrets_file} and MotherDuck environment variable set")
        except Exception as e:
            logger.error(f"Failed to load secrets: {e}")
            raise
    else:
        logger.info("ECHO: MD environment variable already set.")

In [ ]:
set_md_token()

In [ ]:
con_md = duckdb.connect("md:")        # Relies on environment variable being set (by set_md_token)

In [ ]:
con_md.sql("SHOW DATABASES;")         # Show all databases available on MotherDuck

In [ ]:
def check_local_duckdb_database():
    """
    Checks if a local DuckDB database exists. It returns the path to the database if it exists, and None otherwise.
    """
    DATA_DIR = Path("../data")
    EXPERIMENT_TYPE = "RPE"  # or CB
    assert DATA_DIR.exists()
    local_ddb = DATA_DIR / f"Experiments_{EXPERIMENT_TYPE}_2023_Q4.ddb"  # DuckDB database name

    logger.info(f"Database file: {local_ddb.resolve()}")

    if local_ddb.exists():
        return local_ddb
    else:
        return None

In [ ]:
local_ddb = check_local_duckdb_database()

In [ ]:
local_ddb.stem

In [ ]:
# Database names cannot contain hypens (underscores are ok) - also make lower case

def fix_db_name(db_name):
    return db_name.replace("-", "_").lower()

In [ ]:
attach_local_ddb = f"ATTACH '{str(local_ddb)}' AS local_{fix_db_name(local_ddb.stem)};"

In [ ]:
attach_local_ddb

In [ ]:
con_md.sql(attach_local_ddb)

In [ ]:
con_md.sql("SHOW DATABASES;")

In [ ]:

ddb = DuckDBManager(db_path=str(local_ddb), secrets_file=None, logger=logger, dry_run=False)

# dry_run = True can be used to just echo the SQL commands without actually running them i.e. a dry run

In [ ]:
ddb.connect()

In [ ]:
ddb.execute_sql("SHOW ALL TABLES;").df()

In [ ]:
ddb.execute_sql("SELECT * FROM Experiments").df()

In [ ]:
ddb.execute_sql("PRAGMA database_list;").fetchall()

In [ ]:
ddb.get_motherduck_ui_version()

In [ ]:
ddb.get_motherduck_db_version().fetchall()

In [ ]:
ddb.copy_ddb_to_motherduck(str(local_ddb), remote_db_name="tmp_ddb")

In [ ]:
with DuckDBManager(db_path=str(local_ddb), secrets_file=SECRETS_TOML, logger=logger, dry_run=False) as db_manager:
    db_manager.copy_ddb_to_motherduck(local_ddb, remote_db_name="test_class_approach")

## Working with MotherDuck database shared with other users

Example code for this. The database share name is available in the MotherDuck UI (after making the database shareable).

Learn about sharing here: https://motherduck.com/docs/key-tasks/querying-a-shared-motherduck-database.

In [ ]:

# Code for working with MotherDuck shared database

MD_DB_SHARE_NAME = 'my_db_name/cbe96e98-c6b3-4e13-86b4-c3b42cf95aa6'

con_md_share = duckdb.connect()
con_md_share.sql(f"ATTACH 'md:_share/{MD_DB_SHARE_NAME}' AS md_ddb")
con_md_share.sql("USE md_ddb;")

con_md_share.sql("SELECT * FROM Experiments")        # Run an example query
con_md_share.close()